In [1]:
# confirmnando o localhost do min.io
import os
print(os.getenv("MINIO_ENDPOINT"))

None


Comando para reconhecer a pasta na raiz do projeto e conexão com o DuckBD

In [2]:
#os imports somente devem ser adicionados, caso o notebook não reconheça a coinfiguração do pythonpath 
# import os
# import sys

# sys.path.insert(0, os.path.abspath(".."))

from src.utils.database import get_duckdb_connection

con = get_duckdb_connection()


In [3]:
con.execute("""
SELECT COUNT(*) AS total
FROM read_csv(
    's3://raw/censo_escolar/2025/escola_2025.csv',
    delim=';',
    header=True,
    encoding='latin-1',
    ignore_errors=true,
    strict_mode=false,
    all_varchar=true,
    null_padding=true
);
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total
0,214192


In [ ]:
from src.processing.transformacoes import processar_censo_escolar

con = processar_censo_escolar()
con.execute("SELECT COUNT(*) FROM censo_trusted").df()

In [ ]:
con.execute("""
SELECT COUNT(*) AS total
FROM read_csv(
    's3://raw/censo_escolar/2025/escola*.csv',
    delim=';',
    header=True,
    encoding='latin-1',
    ignore_errors=true,
    strict_mode=false,
    all_varchar=true
);
""").df()

In [ ]:
from src.utils.database import get_duckdb_connection

con = get_duckdb_connection()

query = """
SELECT 
    *,
    
    -- extrai o ano do caminho (ex: censo_escolar/2025/...)
    regexp_extract(filename, 'censo_escolar/(\\d{4})/', 1) AS ano

FROM read_csv_auto(
    's3://raw/censo_escolar/*/*.csv',
    
    delim=';',
    header=True,
    encoding='latin-1',    
    -- muito importante pro EDA
    ignore_errors=true,
    filename=true
)

LIMIT 1000
"""

df = con.execute(query).df()

display(df)

In [ ]:
import boto3

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

response = s3.list_objects_v2(Bucket='raw', Prefix='censo_escolar/')

for obj in response.get('Contents', []):
    print(obj['Key'])

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Ajusta o caminho para a raiz do projeto de desigualdade digital
sys.path.append(os.path.abspath(os.path.join('..')))

from src.utils.database import get_duckdb_connection

con = get_duckdb_connection()

# Agora a função s3_ls DEVE ser reconhecida
try:
    df_arquivos = con.execute("SELECT * FROM s3_ls('s3://raw/censo_escolar/')").df()
    display(df_arquivos)
except Exception as e:
    print(f"Erro detalhado: {e}")

In [ ]:
# Teste manual de instalação de extensão
con.execute("INSTALL httpfs; LOAD httpfs;")

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

from src.utils.database import get_duckdb_connection

# Reinicie o Kernel se o erro persistir
con = get_duckdb_connection()

# TESTE DE OURO: Verifica se a extensão httpfs está 'loaded'
status_ext = con.execute("SELECT extension_name, loaded FROM duckdb_extensions() WHERE extension_name = 'httpfs'").df()
print(status_ext)

# Se 'loaded' for True, esse comando ABAIXO não pode falhar:
df_arquivos = con.execute("SELECT * FROM s3_ls('s3://raw/censo_escolar/')").df()
display(df_arquivos)

In [ ]:
BUCKET = "raw"
BASE_PATH = "censo_escolar/2025/"

def load_csv(nome_arquivo):
    local_path = f"data/{nome_arquivo}"
    download_file(f"{BASE_PATH}{nome_arquivo}", BUCKET, local_path)
    return pd.read_csv(local_path, encoding="latin1", low_memory=False)

In [ ]:
escola = load_csv("tabela_escola.csv")

escola.head()

In [ ]:
escola.columns = escola.columns.str.lower().str.strip()
escola = escola.drop_duplicates()

In [ ]:
docente = load_csv("tabela_docente.csv")

docente.columns = docente.columns.str.lower().str.strip()
docente = docente.drop_duplicates()

In [ ]:
matricula = load_csv("tabela_matricula.csv")

matricula.columns = matricula.columns.str.lower().str.strip()
matricula = matricula.drop_duplicates()

In [ ]:
print("Escola:", escola.shape)
print("Docente:", docente.shape)
print("Matrícula:", matricula.shape)

print(escola.isnull().mean().sort_values(ascending=False).head())